### Install Modules

In [10]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -qU transformers accelerate
!pip install -qU bitsandbytes pandas
!pip install -q hf_transfer einops sentence_transformers tensorflow tf-keras

In [2]:
import os
os.environ["JUPYTER_WIDGETS_DISABLED"] = "true"

In [3]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

PyTorch: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8


### Intialize Embedding Generator

In [4]:
import torch,os,gc
from transformers import AutoTokenizer, AutoModel,T5EncoderModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class EmbeddingGenerator:
    def __init__(self,model_name:str="microsoft/codebert-base", chunk_size:int=128, stride:int=68):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.chunk_size = chunk_size
        self.stride = stride
        if self.tokenizer.pad_token is None:
          self.tokenizer.pad_token = self.tokenizer.eos_token

    def clean_memory(self):
        del self.model,self.tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    def generate_embedding(self,code:str):
        all_tokens = self.tokenizer.encode(code, add_special_tokens=False)
        total_length = len(all_tokens)

        if total_length <= self.chunk_size:
            inputs = self.tokenizer(code,return_tensors='pt')
        else:
            chunks = []
            for i in range(0,total_length,self.stride):
                chunk = all_tokens[i:i+self.chunk_size]
                chunks.append(chunk)
            input_ids = [self.tokenizer.build_inputs_with_special_tokens(chunk) for chunk in chunks]
            max_len = max(len(ids) for ids in input_ids)
            attention_masks = []
            padded_input_ids = []
            for chunk in input_ids:
                padding_length = max_len - len(chunk)
                padded_input_ids.append(chunk+[self.tokenizer.pad_token_id]*padding_length)
                attention_masks.append([1]*len(chunk)+[0]*padding_length)
            inputs = {'input_ids': torch.tensor(padded_input_ids), 'attention_mask': torch.tensor(attention_masks)}


        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = self.model(**inputs)

        expanded_attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(outputs.last_hidden_state.shape)
        masked_embeddings = expanded_attention_mask * outputs.last_hidden_state
        code_embedding = (masked_embeddings.sum(1)/ expanded_attention_mask.sum(1)).mean(0)



        if total_length <= self.chunk_size:
            del inputs, outputs, masked_embeddings
        else:
            del inputs,input_ids, attention_masks, outputs, masked_embeddings
        torch.cuda.empty_cache()

        return code_embedding

/root/AI-Pattern-Mining-Project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-12-02 02:00:06.792723: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [5]:
import ast

class DocstringRemover(ast.NodeTransformer):
    """
    An AST node transformer that removes docstrings from Function, Class,
    and Module nodes.
    """
    def _remove_docstring(self, node):
        """A helper to remove the docstring from a node's body."""
        if not node.body:
            return

        # Check if the first statement is a docstring
        if isinstance(node.body[0], ast.Expr) and isinstance(node.body[0].value, ast.Constant):
            # In Python < 3.8, it's ast.Str, but ast.Constant is used for 3.8+
            # This check is sufficient for modern Python versions.
            node.body = node.body[1:]

    def visit_FunctionDef(self, node):
        self._remove_docstring(node)
        self.generic_visit(node) # Visit children nodes
        return node

    def visit_AsyncFunctionDef(self, node):
        self._remove_docstring(node)
        self.generic_visit(node)
        return node

    def visit_ClassDef(self, node):
        self._remove_docstring(node)
        self.generic_visit(node)
        return node

    def visit_Module(self, node):
        self._remove_docstring(node)
        self.generic_visit(node)
        return node

def remove_all_comments(source_code: str) -> str:
    """
    Removes all comments and docstrings from a Python source code string.

    This function leverages the Abstract Syntax Tree (AST) to safely parse
    and rebuild the code without its comments. It correctly handles multi-line
    strings and other complex cases where regex-based solutions might fail.

    Args:
        source_code: A string containing the Python code.

    Returns:
        The Python code with all comments and docstrings removed.
        Returns the original code if it contains a syntax error.

    Requires: Python 3.9+ for ast.unparse()
    """
    try:
        # 1. Parse the source code into an AST.
        #    Hash comments (#) are automatically discarded during this phase.
        tree = ast.parse(source_code)

        # 2. Traverse the tree to remove docstrings.
        remover = DocstringRemover()
        remover.visit(tree)

        # 3. Unparse the modified AST back into source code.
        return ast.unparse(tree)
    except (SyntaxError, TypeError):
        # If the code has a syntax error, parsing will fail.
        print("Could not parse the source code due to a syntax error.")
        return source_code

In [6]:
# @title
import os

def load_code_from_file(file_path:str)->str:
    with open(file_path,"r") as file:
        code = file.read()
        code = remove_all_comments(code)
    return code

def get_all_python_files(repo_path):
    python_files = []
    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(".py"):
                python_files.append(os.path.join(root, file))
    return python_files

def get_all_py_files(directory):
    py_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".py"):
                py_files.append(os.path.join(root, file))
    return py_files

def get_folders(repo_path):
    directories = os.walk(repo_path)
    directories = [i[1] for i in directories][0]
    return directories

### Generate Embeddings

In [11]:
from sentence_transformers import SentenceTransformer
code_rank_embedder = SentenceTransformer("nomic-ai/CodeRankEmbed", trust_remote_code=True)

A new version of the following files was downloaded from https://huggingface.co/nomic-ai/CodeRankEmbed:
- configuration_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/nomic-ai/CodeRankEmbed:
- modeling_hf_nomic_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
<All keys matched successfully>


In [12]:
# @title
import pandas as pd
repo_path = "/root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code"
pattern_embedding_path = "./embeddings_result/embeddings_call_graph_clusters_v3.csv"
pattern_embedding_coderank_path = "./embeddings_result/embeddings_call_graph_clusters_v3_coderank.csv"

embedding_generator = EmbeddingGenerator(model_name="FacebookAI/roberta-base")
embedding_size = embedding_generator.model.config.hidden_size

columns = ["code_file"]+[f'dim_{i}' for i in range(embedding_size)]

if os.path.exists(pattern_embedding_path):
  embeddings_df = pd.read_csv(pattern_embedding_path)
  embeddings_df_2 = pd.read_csv(pattern_embedding_coderank_path)
else:
  embeddings_df = pd.DataFrame(columns=columns)
  embeddings_df_2 = pd.DataFrame(columns=columns)

python_files = get_all_python_files(repo_path)
for file in python_files:
    code = load_code_from_file(file)
    embeddings = embedding_generator.generate_embedding(code)
    code_rank_embeddings = code_rank_embedder.encode([code])[0]
    print("Computed embeddings for file:", file)
    embeddings_df.loc[len(embeddings_df)] = [file]+embeddings.tolist()
    embeddings_df_2.loc[len(embeddings_df_2)] = [file]+code_rank_embeddings.tolist()
embedding_generator.clean_memory()
os.makedirs(os.path.dirname(pattern_embedding_path),exist_ok=True)
embeddings_df.to_csv(pattern_embedding_path,index=False)
embeddings_df_2.to_csv(pattern_embedding_coderank_path,index=False)

Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Token indices sequence length is longer than the specified maximum sequence length for this model (2835 > 512). Running this sequence through the model will result in indexing errors


Computed embeddings for file: /root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code/Advanced MT Prompting/pattern_1.py
Computed embeddings for file: /root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code/Advanced MT Prompting/pattern_10.py
Computed embeddings for file: /root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code/Advanced MT Prompting/pattern_11.py
Computed embeddings for file: /root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code/Advanced MT Prompting/pattern_12.py
Computed embeddings for file: /root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code/Advanced MT Prompting/pattern_13.py
Computed embeddings for file: /root/AI-Pattern-Mining-Project/outputs/prompt & rag/20251028_085918 - Run/generated_code/Advanced MT Prompting/pattern_14.py
Computed embeddings for file: /root/AI-Pattern-Mining-Project/out

In [12]:
len(python_files)

780

In [14]:
embeddings_df.shape

(780, 769)

In [10]:
embeddings_df.head()

,code_file,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,...,dim_758,dim_759,dim_760,dim_761,dim_762,dim_763,dim_764,dim_765,dim_766,dim_767
0,/root/AI-Pattern-Mining-Project/outputs/prompt...,0.010473,-0.086456,0.020704,0.072811,-0.069658,0.079100,0.016109,-0.116357,0.035185,...,0.144655,-0.024537,-0.067877,0.075597,0.099720,0.035907,0.233789,-0.317256,0.070238,0.092933
1,/root/AI-Pattern-Mining-Project/outputs/prompt...,0.002774,-0.155825,-0.016958,0.109435,-0.134882,-0.026956,0.019212,-0.135563,0.029505,...,0.080175,-0.018589,-0.053212,0.127422,0.065531,0.024898,0.390377,-0.205732,0.016368,0.122040
2,/root/AI-Pattern-Mining-Project/outputs/prompt...,-0.000473,-0.199095,-0.024379,0.095847,-0.138792,-0.012554,0.002148,-0.176035,0.042417,...,0.129090,-0.024046,-0.045352,0.143503,0.057276,0.029717,0.420159,-0.237789,0.003915,0.154167
3,/root/AI-Pattern-Mining-Project/outputs/prompt...,0.016031,-0.193469,0.001521,0.100525,-0.102650,-0.121538,0.017241,-0.197112,0.069094,...,0.092082,0.019782,-0.054770,0.114624,0.046622,0.047399,0.395274,-0.168212,-0.044403,0.211058
4,/root/AI-Pattern-Mining-Project/outputs/prompt...,0.003953,-0.221190,-0.005675,0.139310,-0.175301,-0.049305,0.008009,-0.185112,0.046989,...,0.100162,-0.029435,-0.020810,0.141135,0.036710,0.048247,0.451174,-0.169562,-0.032311,0.211697


### Call Graph to Embeddings

In [ ]:
from torch_geometric.nn import Graph
from torch_geometric.data import Data
import os,ast
import pandas as pd
from networkx import nx


/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'Graph2Vec' from 'torch_geometric.nn' (/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/torch_geometric/nn/__init__.py)

In [ ]:
class CallGraphVisitor(ast.NodeVisitor):
    def __init__(self, filepath, global_functions, base_dir):
        self.filepath = os.path.relpath(filepath, base_dir)
        self.base_dir = base_dir
        self.global_functions = global_functions

        self.current_function = None
        self.current_class = None
        self.calls = []
        self.functions = []
        self.imports = {}  # alias -> file path

    # --- Import tracking ---
    def visit_Import(self, node):
        for alias in node.names:
            module_name = alias.name  # e.g., "utils.helper"
            asname = alias.asname or module_name.split(".")[-1]
            module_path = self._resolve_module_to_path(module_name)
            if module_path:
                self.imports[asname] = module_path
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        # e.g., from example02 import y
        if node.module:
            module_name = node.module
            module_path = self._resolve_module_to_path(module_name)
            for alias in node.names:
                asname = alias.asname or alias.name
                self.imports[asname] = module_path
        self.generic_visit(node)

    def _resolve_module_to_path(self, module_name):
        """Convert module name to file path if exists in the project."""
        path_guess = os.path.join(self.base_dir, *module_name.split(".")) + ".py"
        if os.path.exists(path_guess):
            return os.path.relpath(path_guess, self.base_dir)
        return None

    # --- Class and function tracking ---
    def visit_ClassDef(self, node):
        prev_class = self.current_class
        self.current_class = node.name
        self.generic_visit(node)
        self.current_class = prev_class

    def visit_FunctionDef(self, node):
        if self.current_class:
            full_name = f"{self.filepath}:{self.current_class}.{node.name}"
            short_name = f"{self.current_class}.{node.name}"
        else:
            full_name = f"{self.filepath}:{node.name}"
            short_name = node.name

        self.functions.append({
            "full_name": full_name,
            "short_name": short_name,
            "class_name": self.current_class,
            "filepath": self.filepath,
            "lineno": node.lineno,
            "col_offset": node.col_offset
        })

        self.global_functions.setdefault(short_name, []).append(full_name)

        prev_function = self.current_function
        self.current_function = full_name
        self.generic_visit(node)
        self.current_function = prev_function

    # --- Call tracking ---
    def visit_Call(self, node):
        # If not inside a function, treat it as a top-level (module) call
        caller = self.current_function or f"{self.filepath}:<module>"

        callee_full = None

        # Case 1: direct call like y()
        if isinstance(node.func, ast.Name):
            name = node.func.id
            if name in self.imports and self.imports[name]:
                callee_file = self.imports[name]
                matches = self.global_functions.get(name, [])
                if matches:
                    callee_full = matches[0].replace(matches[0].split(":")[0], callee_file)
                else:
                    callee_full = f"{callee_file}:{name}"
            elif name in self.global_functions:
                callee_full = self.global_functions[name][0]
            else:
                callee_full = name

        # Case 2: attribute call like mod.func()
        elif isinstance(node.func, ast.Attribute):
            value = getattr(node.func.value, "id", None)
            attr = node.func.attr
            if value and value in self.imports and self.imports[value]:
                callee_file = self.imports[value]
                callee_full = f"{callee_file}:{attr}"
            else:
                callee_full = attr

        if callee_full:
            self.calls.append((caller, callee_full))

        self.generic_visit(node)


In [ ]:

import networkx as nx

G = nx.read_edgelist("call_graph.csv", delimiter=",")

node2vec = Node2Vec(G, dimensions=128, walk_length=10, num_walks=200)
model = node2vec.fit(window=5)

# get embedding for any function
vec = model.wv["train.py:GeneratorFullModel.forward"]
print(vec)
